In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Quantitative Evaluation Results

In [12]:
# Install required libraries
!pip install -q datasets huggingface_hub

# Import libraries
import pandas as pd
import os
from datasets import Dataset
from huggingface_hub import login

# Path to folder containing metric CSV files
DATA_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/Contex_Summary_Semantic_Lexical_Similarity_Results"

# Read all CSV files and convert metric rows -> columns
all_rows = []

for file in os.listdir(DATA_DIR):

    if file.endswith(".csv"):

        file_path = os.path.join(DATA_DIR, file)

        df = pd.read_csv(file_path)

        # Each file contains rows like: model | metric | value
        model_name = df["model"].iloc[0]

        row = {"model": model_name}

        for _, r in df.iterrows():

            metric_name = r["metric"]
            metric_value = r["value"]

            row[metric_name] = metric_value

        all_rows.append(row)

# Combine all models into one dataframe
combined_df = pd.DataFrame(all_rows)

combined_df = combined_df.sort_values("model").reset_index(drop=True)

# Round all numeric values to 4 decimal places
combined_df = combined_df.round(4)

print("\nCombined Metrics Table\n")

display(combined_df)


Combined Metrics Table



,model,BLEU,Google BLEU,METEOR,ROUGE-1,ROUGE-2,ROUGE-L,Cosine Similarity (mean),BERTScore F1 (mean),BARTScore (mean)
0,Llama-3.1_8B-instruct,0.3791,0.3929,0.6175,0.6751,0.4351,0.5486,0.9264,0.9346,-1.8701
1,Llama-3.2_3B-instruct,0.3663,0.3788,0.5976,0.6634,0.4150,0.5305,0.9255,0.9321,-1.9362
2,Phi-4-mini,0.3620,0.3712,0.5804,0.6512,0.4039,0.5159,0.9147,0.9309,-1.9466
3,Qwen3-4B-instruct,0.3753,0.3839,0.6000,0.6636,0.4191,0.5332,0.9201,0.9324,-1.9236
4,Qwen3-8B-instruct,0.4021,0.4069,0.6179,0.6819,0.4478,0.5609,0.9286,0.9363,-1.8455
5,gemini-2.5-flash,0.2652,0.2977,0.4730,0.5912,0.3354,0.4511,0.8656,0.9194,-2.1386
6,gpt-4.1,0.2998,0.3277,0.5470,0.6238,0.3507,0.4706,0.9117,0.9241,-2.1292


In [13]:
save_path = "/content/drive/MyDrive/combined_model_metrics.csv"

combined_df.to_csv(save_path, index=False)

print("\nSaved combined table to:", save_path)

# Push dataset to HuggingFace

login()

dataset = Dataset.from_pandas(combined_df)

dataset.push_to_hub("Lakshan2003/Contex_Summary_Semantic_Lexical_Similarity_Results")

print("\nDataset uploaded to HuggingFace successfully")


Saved combined table to: /content/drive/MyDrive/combined_model_metrics.csv


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.05kB / 5.05kB            


Dataset uploaded to HuggingFace successfully


## LLM as a Judge Results

In [10]:
from datasets import load_dataset, Dataset
import pandas as pd

datasets_list = [
    "Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data",
    "Lakshan2003/GPT-4.1-customerservice-context-summarization-llm-judge-data",
    "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-llm-judge-data",
    "Lakshan2003/Llama3.2-3B-instruct-customerservice-context-summarization-llm-judge-data",
    "Lakshan2003/Phi-4-mini-customerservice-context-summarization-llm-judge-data",
    "Lakshan2003/Qwen3-4B-customerservice-context-summarization-llm-judge-data",
    "Lakshan2003/Qwen3-8B-customerservice-context-summarization-llm-judge-data"
]

results = []

for repo in datasets_list:

    print(f"Loading {repo}")

    ds = load_dataset(repo, split="train")
    df = ds.to_pandas()

    df["Information-Accuracy"] = pd.to_numeric(df["Information-Accuracy"], errors="coerce")
    df["Structural-Clarity"] = pd.to_numeric(df["Structural-Clarity"], errors="coerce")
    df["Faithfulness"] = pd.to_numeric(df["Faithfulness"], errors="coerce")

    avg_info = df["Information-Accuracy"].mean()
    avg_structure = df["Structural-Clarity"].mean()
    avg_faithfulness = df["Faithfulness"].mean()
    overall = (avg_info + avg_structure + avg_faithfulness) / 3

    model_name = repo.split("/")[1].replace(
        "-customerservice-context-summarization-llm-judge-data", ""
    )

    results.append({
        "Model": model_name,
        "Information-Accuracy": round(avg_info, 4),
        "Structural-Clarity": round(avg_structure, 4),
        "Faithfulness": round(avg_faithfulness, 4),
        "Overall-Qualitative-Score": round(overall, 4)
    })

results_df = pd.DataFrame(results)

Loading Lakshan2003/gemini-2.5-flash-customerservice-context-summarization-llm-judge-data


data/train-00000-of-00001.parquet:   0%|          | 0.00/1.97M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading Lakshan2003/GPT-4.1-customerservice-context-summarization-llm-judge-data


data/train-00000-of-00001.parquet:   0%|          | 0.00/2.03M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading Lakshan2003/Qwen3-8B-customerservice-context-summarization-llm-judge-data


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.45M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [11]:
results_df

,Model,Information-Accuracy,Structural-Clarity,Faithfulness,Overall-Qualitative-Score
0,gemini-2.5-flash,4.7930,4.9300,4.7700,4.8310
1,GPT-4.1,4.8060,4.9480,4.7510,4.8350
2,Llama3.1-8b-instruct,4.6897,4.9039,4.6296,4.7411
3,Llama3.2-3B-instruct,4.5816,4.8398,4.4825,4.6346
4,Phi-4-mini,4.6300,4.8760,4.5390,4.6817
5,Qwen3-4B,4.6580,4.8760,4.5640,4.6993
6,Qwen3-8B,4.6927,4.8959,4.6256,4.7381


In [14]:
results_dataset = Dataset.from_pandas(results_df)

results_dataset.push_to_hub(
    "Lakshan2003/context-summarization-llm-judge-results",
    private=False
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 2.86kB / 2.86kB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/context-summarization-llm-judge-results/commit/5924b8f4c1835fea867bd172a6c0f6de15d265d1', commit_message='Upload dataset', commit_description='', oid='5924b8f4c1835fea867bd172a6c0f6de15d265d1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/context-summarization-llm-judge-results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/context-summarization-llm-judge-results'), pr_revision=None, pr_num=None)